# 04 — Ajustar o comportamento sem substituir o RAG

O RAG da etapa 03 busca conhecimento no momento da pergunta. QLoRA atende a outro objetivo: adaptar padrões de resposta com exemplos supervisionados, treinando adaptadores pequenos enquanto a base quantizada permanece congelada. O encoder e o índice Chroma não são modificados.

```mermaid
flowchart LR
 A[Qwen base em 4 bits] --> C[base congelada + adaptadores LoRA]
 B[exemplos sintéticos revisáveis: pergunta → resposta] --> D[loss nos tokens da resposta]
 D --> C
 C --> E[adaptador salvo]
 E --> F[etapa 05: avaliar com/sem RAG]
```

O `dataset_sft.jsonl` contém 1.000 exemplos sintéticos: 800 para treino, 100 para validação e 100 para teste, distribuídos entre cinco famílias temáticas. Eles foram criados para exercitar conceitos e cenários numéricos; não foram extraídos dos PDFs, cujo catálogo não autoriza uso em treinamento. O conjunto precisa de revisão técnica e avaliação humana antes de qualquer conclusão sobre especialização.

In [5]:
# Objetivo: definir diretórios do ajuste.
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

Projeto: d:\Projetos_v2\Agente_IA_Transformers_Credito


## 1. Carregar uma base limpa

O ajuste começa da mesma revisão do Qwen usada como baseline na etapa 00, sem adaptador previamente carregado. Isso torna o ponto de partida reproduzível e evita continuar acidentalmente um treino anterior. O modelo é carregado em 4 bits na GPU; por isso, deve-se encerrar a interface antes do treino para liberar memória.

A quantização reduz a memória usada para armazenar os pesos da base. Ela não elimina o custo de ativações, gradientes dos adaptadores e estados do otimizador, que também dependem do tamanho do lote e da sequência.

In [6]:
# Objetivo: carregar o Qwen quantizado como base.
import random
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

NOME_MODELO = "Qwen/Qwen2.5-3B-Instruct"
INSTRUCAO_SISTEMA = (
    "Responda em português, de forma didática e impessoal. "
    "Diferencie fatos, hipóteses e exemplos sintéticos. Não invente resultados."
)
set_seed(RANDOM_STATE)
# Carrega o tokenizador compatível com o checkpoint base.
tokenizador = AutoTokenizer.from_pretrained(NOME_MODELO)
# Carrega a base causal quantizada; device_map escolhe GPU, dtype fixa precisão e attn_implementation seleciona backend.
modelo = AutoModelForCausalLM.from_pretrained(
    NOME_MODELO,  # Checkpoint base que será ajustado.
    device_map={"": 0},  # Mapeia o modelo para a primeira GPU CUDA.
    dtype=torch.float16,  # Precisão dos módulos não quantizados.
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,  # Armazena pesos da base em 4 bits.
        bnb_4bit_quant_type="nf4",  # Usa o formato de quantização NF4.
        bnb_4bit_use_double_quant=True,  # Quantiza constantes para economizar memória.
        bnb_4bit_compute_dtype=torch.float16,  # Precisão usada nos cálculos da base.
    ),
    attn_implementation="sdpa",  # Backend de atenção otimizado do PyTorch.
)
print("Base limpa carregada.")

Loading weights: 100%|██████████| 434/434 [00:02<00:00, 161.37it/s]


Base limpa carregada.


## 2. Preparar exemplos e conferir a supervisão

O `dataset_sft.jsonl` contém perguntas e respostas sintéticas organizadas por família e subtema. A divisão é estratificada por família: os cinco temas aparecem em treino, validação e teste, sem repetir a mesma pergunta. Isso permite comparar perguntas diferentes sobre temas conhecidos; não mede generalização para uma família temática inteiramente nova. Os PDFs do corpus estão marcados como não autorizados para treinamento e não foram usados como fonte desses exemplos.

### O que a loss ensina

Um modelo causal prevê o próximo token dado o contexto anterior. Para a sequência `pergunta → resposta A → resposta B → EOS`, a previsão de `A` usa a pergunta como contexto; a previsão de `B` usa a pergunta e `A`. Com supervisão apenas da resposta, as posições do prompt recebem label `-100`, que a função de loss ignora. Os tokens da resposta, incluindo o fim da sequência quando presente, são supervisionados.

$$
L = -\frac{1}{N}\sum_{t:y_t\ne -100}\log p_\theta(y_t\mid x_{<t}),
$$

em que $N$ é o número de tokens supervisionados. Em Transformers, o modelo causal desloca logits e labels internamente para alinhar cada previsão ao próximo token. Deslocar labels também manualmente duplicaria esse ajuste. A inspeção de tokens e labels confirma que a pergunta continua como contexto e que a loss se aplica à resposta esperada.

As perguntas são únicas entre os splits, mas exemplos de um mesmo subtema podem aparecer em treino e avaliação. Portanto, a validação mede respostas a novos cenários e formulações dentro dos temas cobertos, não conhecimento independente de temas. Como os exemplos são sintéticos, a avaliação ainda exige revisão humana e não demonstra desempenho em produção.

As chamadas usam `return_dict=False` para obter uma lista simples de IDs. Em versões recentes de Transformers, o padrão pode ser um `BatchEncoding`; nesse objeto, `len(...)` conta campos como `input_ids` e `attention_mask`, e não o número de tokens.

In [7]:
# Objetivo: preparar exemplos e máscaras de loss.
exemplos = [json.loads(linha) for linha in (DIRETORIO_PROJETO / "dataset_sft.jsonl").read_text(encoding="utf-8").splitlines() if linha.strip()]
treino = [e for e in exemplos if e["split"] == "treino"]
validacao = [e for e in exemplos if e["split"] == "validacao"]
teste = [e for e in exemplos if e["split"] == "teste"]
print("Treino/validação/teste:", len(treino), len(validacao), len(teste))
perguntas_unicas = {e["pergunta"] for e in exemplos}
assert len(perguntas_unicas) == len(exemplos), "Há perguntas duplicadas entre os splits."
familias_treino = {e["familia"] for e in treino}
familias_validacao = {e["familia"] for e in validacao}
familias_teste = {e["familia"] for e in teste}
assert familias_treino == familias_validacao == familias_teste, "A divisão deve cobrir as mesmas famílias em todos os splits."
print(treino[0])

lotes = {"treino": [], "validacao": []}
for nome_split, registros in [("treino", treino), ("validacao", validacao)]:
    for exemplo in registros:
        mensagens = [{"role": "system", "content": INSTRUCAO_SISTEMA}, {"role": "user", "content": exemplo["pergunta"]}]
        # tokenize=True retorna IDs; add_generation_prompt abre o turno do assistente.
        # return_dict=False força uma lista de IDs e mantém len(prefixo) como quantidade de tokens.
        prefixo = tokenizador.apply_chat_template(
            mensagens,  # Instrução de sistema e pergunta, sem resposta-alvo.
            tokenize=True,  # Retorna IDs de tokens em vez do texto formatado.
            add_generation_prompt=True,  # Acrescenta o início do turno do assistente.
            return_dict=False,  # Garante uma lista simples de IDs para len(prefixo).
        )
        # A sequência completa inclui a resposta que será supervisionada no treino.
        completo = tokenizador.apply_chat_template(
            mensagens + [{"role": "assistant", "content": exemplo["resposta"]}],  # Conversa mais resposta correta.
            tokenize=True,  # Produz IDs para alimentar o modelo.
            return_dict=False,  # Mantém retorno como lista de IDs, sem BatchEncoding.
        )
        assert completo[:len(prefixo)] == prefixo
        if len(completo) > 512:
            raise ValueError(f"Exemplo longo: {exemplo['familia']}; revisar antes do treino.")
        labels = [-100] * len(prefixo) + completo[len(prefixo):]
        assert any(x != -100 for x in labels)
        lotes[nome_split].append({"input_ids": torch.tensor([completo]), "attention_mask": torch.ones(1, len(completo), dtype=torch.long), "labels": torch.tensor([labels])})
print("Shape:", lotes["treino"][0]["input_ids"].shape)
print("Labels:", lotes["treino"][0]["labels"])
print("Texto supervisionado:", tokenizador.decode([x for x in lotes["treino"][0]["labels"][0].tolist() if x != -100]))

Treino/validação/teste: 800 100 100
{'familia': 'transformers', 'subtema': 'causal_mask', 'pergunta': 'Uma sequência causal é treinada para prever o próximo token. O que a máscara triangular impede em relação a saldo observado após a data de concessão?', 'resposta': 'A máscara causal bloqueia posições posteriores na sequência para que cada token use apenas o prefixo disponível. Ela evita vazamento do alvo dentro da tarefa de linguagem. Isso não substitui controles de leakage em crédito: também é necessário verificar se cada variável, como saldo observado após a data de concessão, existiria na data de decisão.', 'split': 'treino', 'origem': 'sintético; cenário gerado para estudo, não extraído dos PDFs; revisar tecnicamente antes do treino'}
Shape: torch.Size([1, 175])
Labels: tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -

## 3. Congelar a base e inserir LoRA

Uma camada linear aplica pesos $W$ às entradas. Treinar todos os valores de $W$ pode ser caro. LoRA mantém $W$ congelada e aprende uma atualização de baixo posto:

$$
W'=W+\frac{\alpha}{r}BA,
\qquad A\in\mathbb{R}^{r\times d_{in}},\quad B\in\mathbb{R}^{d_{out}\times r}.
$$

Para uma matriz $1024\times1024$, há 1.048.576 pesos. Com rank $r=8$, as matrizes $A$ e $B$ têm juntas $8\times1024+1024\times8=16.384$ parâmetros treináveis, aproximadamente 1,56% do total daquela matriz. É uma ilustração de uma camada, não do adaptador inteiro nem da redução total de memória.

### O que QLoRA acrescenta

QLoRA mantém os pesos da base quantizados e congelados enquanto treina os adaptadores LoRA. Nesta configuração, a base usa 4 bits com NF4 e computação FP16; LoRA usa rank 8, escala `alpha=16` e atua em `q_proj` e `v_proj`. Essas escolhas limitam o ajuste às projeções de consulta e valor da atenção. Quantização e poucos parâmetros treináveis reduzem parte do custo, mas não removem o custo de ativações e do otimizador.

A execução fica desativada por padrão (`EXECUTAR_AJUSTE=False`) para permitir revisar dataset e tensores antes de usar a GPU. A flag é uma proteção operacional; o conjunto pequeno continua sendo um limite científico mesmo que o código execute sem erro.

In [10]:
# Objetivo: configurar adaptadores LoRA.
EXECUTAR_AJUSTE = True  # Alterar para True após revisar os exemplos e encerrar o chat.
if EXECUTAR_AJUSTE:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if hasattr(modelo, "peft_config"):
        raise RuntimeError("Recarregue a base limpa antes de repetir o ajuste.")
    modelo.config.use_cache = False
    # Prepara a base quantizada para treino de adaptadores; checkpointing reduz memória recomputando ativações.
    modelo = prepare_model_for_kbit_training(modelo, use_gradient_checkpointing=True)
    config_lora = LoraConfig(
        r=8,  # Rank controla a dimensão das atualizações de baixo posto.
        lora_alpha=16,  # Escala aplicada às atualizações LoRA.
        lora_dropout=0.05,  # Dropout nas entradas do adaptador durante o treino.
        target_modules=["q_proj", "v_proj"],  # Projeções de query e value que recebem adaptadores.
        bias="none",  # Não adiciona parâmetros de bias treináveis.
        task_type="CAUSAL_LM",  # Define a tarefa como geração causal de texto.
        base_model_name_or_path=NOME_MODELO,  # Registra o checkpoint base associado.
        revision=getattr(modelo.config, "_commit_hash", None),  # Registra o commit efetivamente carregado.
    )
    # Acopla ao modelo base o adaptador descrito pela configuração LoRA.
    modelo = get_peft_model(modelo, config_lora)
    modelo.print_trainable_parameters()
else:
    print("Ajuste desativado. Nenhum peso foi treinado.")

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


## 4. Treinar, validar e salvar o melhor adaptador

O loop explicita as etapas de otimização: forward calcula a saída e a loss; backward calcula gradientes dos parâmetros treináveis; acumulação soma gradientes de microbatches; AdamW atualiza os adaptadores. Com batch de um exemplo e acumulação de quatro, ocorre uma atualização após cada grupo de quatro exemplos (ou após o grupo final menor). Isso reduz a memória por passo, embora não transforme exemplos diferentes em um lote processado simultaneamente.

A cada época, a loss de validação é calculada sem atualizar pesos e ponderada pela quantidade de tokens supervisionados. O adaptador com menor loss de validação é salvo; o conjunto de teste não participa dessa escolha. Exemplo apenas ilustrativo: losses 0,90, 0,72 e 0,78 apontariam a segunda época como checkpoint. Loss menor indica melhor previsão dos tokens de referência nessa validação, mas não prova que explicações, citações ou uso de evidências melhoraram.

Com poucas famílias e poucos exemplos, a validação é muito instável e serve sobretudo para depuração. Após salvar, base e adaptador devem ser recarregados em um processo limpo para conferir compatibilidade. A decisão de oferecer o modo ajustado depende da comparação de respostas da etapa 05, não apenas da loss.

In [11]:
# Objetivo: treinar, validar e salvar o melhor adaptador.
if EXECUTAR_AJUSTE:
    import random
    set_seed(RANDOM_STATE)
    # AdamW atualiza somente parâmetros treináveis; lr controla o tamanho das atualizações.
    otimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=1e-4)
    EPOCAS = 3
    ACUMULACAO = 4
    melhor_loss = float("inf")
    historico_treino = []
    for epoca in range(EPOCAS):
        modelo.train()
        ordem = list(range(len(lotes["treino"])))
        random.Random(RANDOM_STATE + epoca).shuffle(ordem)
        perdas = []
        for inicio in range(0, len(ordem), ACUMULACAO):
            grupo = ordem[inicio:inicio + ACUMULACAO]
            otimizador.zero_grad(set_to_none=True)
            for indice in grupo:
                batch = {k:v.to("cuda:0") for k,v in lotes["treino"][indice].items()}
                # O modelo calcula a loss a partir de input_ids, attention_mask e labels do lote.
                saida = modelo(**batch)
                if not torch.isfinite(saida.loss):
                    raise RuntimeError("Loss não finita. Interromper e investigar precisão/dados.")
                perdas.append(saida.loss.item())
                (saida.loss / len(grupo)).backward()
            # Limita a norma dos gradientes para reduzir atualizações excessivas.
            torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.0)
            otimizador.step()
        modelo.eval()
        soma_loss, total_tokens = 0.0, 0
        with torch.no_grad():
            for lote in lotes["validacao"]:
                batch = {k:v.to("cuda:0") for k,v in lote.items()}
                quantidade = int((batch["labels"][:, 1:] != -100).sum())
                soma_loss += modelo(**batch).loss.item() * quantidade
                total_tokens += quantidade
        loss_val = soma_loss / total_tokens
        registro_epoca = {"epoca": epoca+1, "loss_treino": sum(perdas)/len(perdas), "loss_validacao": loss_val}
        historico_treino.append(registro_epoca)
        print(registro_epoca)
        if loss_val < melhor_loss:
            melhor_loss = loss_val
            # Salva os pesos do adaptador no diretório escolhido; não duplica a base quantizada.
            modelo.save_pretrained(DIRETORIO_PROJETO / "adaptador_pf")
            tokenizador.save_pretrained(DIRETORIO_PROJETO / "adaptador_pf")
    (DIRETORIO_RESULTADOS / "historico_qlora.json").write_text(json.dumps(historico_treino, indent=2), encoding="utf-8")
    print("Melhor adaptador salvo. Reiniciar o kernel/servidor para carregá-lo na inferência.")

c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'epoca': 1, 'loss_treino': 1.6571595146879554, 'loss_validacao': 0.7807387895901763}
{'epoca': 2, 'loss_treino': 0.32959889685269445, 'loss_validacao': 0.16437656681344998}
{'epoca': 3, 'loss_treino': 0.09747475846786983, 'loss_validacao': 0.10935790230518794}
Melhor adaptador salvo. Reiniciar o kernel/servidor para carregá-lo na inferência.
